# Rental proximity to all destination types

Computes road distance from every rental property to its nearest
facility, for each destination type listed in `DESTINATION_SETS`. For
each type: find the nearest candidates by straight-line distance, route
to them with OpenRouteService, and confirm the result is provably the
true nearest. Locations still unconfirmed after a second, wider pass
are left as `NaN` for manual investigation, rather than resolved
automatically.

## Imports

In [1]:
import getpass
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from sklearn.neighbors import BallTree

## Configuration

Every constant that changes per destination lives in `DESTINATION_SETS`;
everything else is shared across all of them.

In [2]:
RENTALS_PATH = "../../rea_data/curated/vic_rentals_curated_final.parquet"
RENTAL_LAT_COLUMN = "lat"
RENTAL_LON_COLUMN = "lon"

SCHOOLS_PATH = "../../external_data/schools.csv"
PARKRES_PATH = "../../external_data/parks_parkres.csv"

DESTINATION_SETS = {
    "sports_rec": "../../external_data/sports_rec_facilities_exact.csv",
    "shopping_centre": "../../external_data/shopping_centres.csv",
    "medical_doctors": "../../external_data/doctors_clinics.csv",
    "medical_clinic": "../../external_data/specialised_clinics.csv",
    "hospital": "../../external_data/hospitals.csv",
    "park_reserve": "../../external_data/parkres_local_reserve.csv",
    "park_national": "../../external_data/parkres_major_park.csv",
    "park_local": "../../external_data/parks_local_park.csv",
    "school_primary": "../../external_data/primary_school.csv",
    "school_secondary": "../../external_data/secondary_school.csv",
    "bus_stops_PTV": "../../external_data/ptv_stops_bus.csv",
    "train_stations_PTV": "../../external_data/ptv_stops_train.csv"
}

In [3]:
SCHOOL_TYPE_GROUPS = {
    "primary_school": ["Primary", "Pri/Sec"],
    "secondary_school": ["Secondary", "Pri/Sec"]
}

PARKRES_CATEGORIES = ["major_park", "local_reserve"]

In [4]:
N_CANDIDATES = 10
N_CANDIDATES_SECOND_PASS = 25
N_CANDIDATES_THIRD_PASS = 50
EARTH_RADIUS_M = 6_371_000
STRIP_DEGREES = 0.05

In [5]:
ORS_URL = "https://api.heigit.org/openrouteservice/v2/matrix"
ORS_PROFILE = "driving-car"
MAX_MATRIX_CELLS = 3_500
MAX_REQUESTS_PER_RUN = 450
SECONDS_BETWEEN_REQUESTS = 1.6
RETRY_STATUS_CODES = {403, 500, 502, 503, 504}
RETRY_WAIT_SECONDS = [10, 30, 60]

In [6]:
OUTPUT_DIR = Path("proximity_results")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)

## Split schools and parks into sub-categories

`schools.csv` and the parks files hold several categories in one file
(`School_Type`, `PARK_CATEGORY`). Each category needed as its own
proximity destination is filtered out and saved separately here, before
the pipeline runs.

In [7]:
def filter_and_save(df, column, value, output_path):
    """Save the rows of df where column equals value to output_path.

    Returns the number of rows saved, so the caller can check the
    filter matched something.
    """
    filtered = df[df[column] == value]
    filtered.to_csv(output_path, index=False)
    return len(filtered)

### Schools by type

`School_Type` includes "Pri/Sec" schools that offer both levels; these
are included in both the primary and secondary files, since a Pri/Sec
school is a genuine option for either age group.

In [8]:
schools = pd.read_csv(SCHOOLS_PATH)
print(schools["School_Type"].value_counts())

School_Type
Primary      1567
Secondary     360
Pri/Sec       248
Special       117
Language        5
Name: count, dtype: int64


In [9]:
school_paths = {}

for destination_name, school_types in SCHOOL_TYPE_GROUPS.items():
    output_path = f"../../external_data/{destination_name}.csv"
    filtered = schools[schools["School_Type"].isin(school_types)]
    filtered.to_csv(output_path, index=False)
    school_paths[destination_name] = output_path
    print(f"{destination_name}: {len(filtered)} rows -> {output_path}")

primary_school: 1815 rows -> ../../external_data/primary_school.csv
secondary_school: 608 rows -> ../../external_data/secondary_school.csv


In [10]:
def standardise_school_columns(path):
    """Rename school columns to match the other facility files."""
    schools = pd.read_csv(path)
    schools = schools.rename(columns={
        "School_No": "Facility ID",
        "school_lat": "Latitude",
        "school_lon": "Longitude",
    })
    schools.to_csv(path, index=False)

for output_path in school_paths.values():
    standardise_school_columns(output_path)

### Parks by category

`parks_parkres.csv` holds `major_park` and `local_reserve` in one file;
`parks_local_park.csv` is already its own category (`local_park`) from
the Overpass extract, so it only needs a path, not filtering.

In [11]:
parkres = pd.read_csv(PARKRES_PATH)
print(parkres["PARK_CATEGORY"].value_counts())

PARK_CATEGORY
local_reserve    6309
major_park       2486
Name: count, dtype: int64


In [12]:
park_paths = {}

for category in PARKRES_CATEGORIES:
    output_path = f"../../external_data/parkres_{category}.csv"
    n_rows = filter_and_save(
        parkres, "PARK_CATEGORY", category, output_path
    )
    park_paths[category] = output_path
    print(f"{category}: {n_rows} rows -> {output_path}")

major_park: 2486 rows -> ../../external_data/parkres_major_park.csv
local_reserve: 6309 rows -> ../../external_data/parkres_local_reserve.csv


## Load the rental properties

Many rentals share the same coordinates, and identical coordinates
always have identical distances, so distance work is done once per
unique location and joined back to every property at the end.

In [13]:
rentals_raw = pd.read_parquet(RENTALS_PATH)

print(f"{len(rentals_raw)} properties")

12317 properties


In [14]:
def get_unique_locations(rentals, lat_column, lon_column):
    """Return each distinct (lat, lon) pair in rentals, once."""
    coordinates = rentals[[lat_column, lon_column]].dropna()
    return coordinates.drop_duplicates().reset_index(drop=True)


unique_locations = get_unique_locations(
    rentals_raw, RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN
)
location_radians = np.radians(
    unique_locations[[RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN]].to_numpy()
)

print(f"{len(unique_locations)} unique locations")

11007 unique locations


## Load the API key

Read from the `ORS_API_KEY` environment variable (via a `.env` file),
or prompted for if not found, so it is never saved in the notebook.

In [16]:
def load_ors_api_key():
    """Load the ORS API key from .env, or prompt for it if missing."""
    load_dotenv()
    api_key = os.environ.get("ORS_API_KEY", "").strip()
    if not api_key:
        api_key = getpass.getpass("Enter your ORS API key: ").strip()
    return api_key


ORS_API_KEY = load_ors_api_key()
print(f"Key length: {len(ORS_API_KEY)} characters")

Key length: 120 characters


## Batching: pack locations into ORS matrix requests

A matrix request returns every source against every destination, so its
size is sources x destinations. Grouping geographic neighbours together
keeps each batch's destination list small, since neighbours share most
of their candidate facilities.

In [17]:
def sort_locations_into_strips(locations, lat_column, lon_column, strip_degrees):
    """Order location row numbers so that neighbours are adjacent.

    Cuts the map into horizontal strips of latitude and sorts within
    each strip by longitude, so nearby locations end up next to each
    other in the returned order.
    """
    strip_number = np.floor(locations[lat_column] / strip_degrees)
    ordered = (
        locations.assign(strip_number=strip_number)
        .sort_values(["strip_number", lon_column])
    )
    return ordered.index.to_numpy()

In [18]:
def build_batches(order, candidates, max_cells):
    """Group locations into batches whose ORS matrix fits in max_cells.

    order: location row numbers, arranged so that neighbours are
    adjacent. candidates: array whose row i holds the facility
    positions for location i. Returns a list of
    (location_rows, facility_positions) pairs.
    """
    batches = []
    sources = []
    destinations = set()

    for location in order:
        location_candidates = set(candidates[location].tolist())
        merged = destinations | location_candidates
        matrix_cells = (len(sources) + 1) * len(merged)

        if sources and matrix_cells > max_cells:
            batches.append((sources, sorted(destinations)))
            sources = []
            merged = location_candidates

        sources.append(int(location))
        destinations = merged

    if sources:
        batches.append((sources, sorted(destinations)))
    return batches

## Send one ORS matrix request, with retries

`request_matrix` sends a single request. `request_matrix_with_retries`
wraps it, retrying temporary server errors (a 403 that other ORS users
report as intermittent, plus 5xx errors) after a short wait.

In [19]:
def build_matrix_request_body(property_points, facility_points):
    """Build the JSON body for one ORS matrix request.

    Both point arrays hold [longitude, latitude] rows.
    """
    n_sources = len(property_points)
    n_destinations = len(facility_points)
    all_points = np.vstack([property_points, facility_points]).tolist()
    return {
        "locations": all_points,
        "sources": list(range(n_sources)),
        "destinations": list(
            range(n_sources, n_sources + n_destinations)
        ),
        "metrics": ["distance", "duration"],
        "units": "m",
    }

In [20]:
def request_matrix(property_points, facility_points, api_key):
    """Ask ORS for road distances and durations for one batch.

    Returns two arrays shaped (n_properties, n_facilities), with NaN
    where ORS found no route.
    """
    body = build_matrix_request_body(property_points, facility_points)
    response = requests.post(
        f"{ORS_URL}/{ORS_PROFILE}",
        json=body,
        headers={"Authorization": api_key},
        timeout=120,
    )
    response.raise_for_status()
    result = response.json()
    distances = np.array(result["distances"], dtype=float)
    durations = np.array(result["durations"], dtype=float)
    return distances, durations

In [21]:
def request_matrix_with_retries(property_points, facility_points, api_key):
    """Call request_matrix, retrying temporary errors after a wait."""
    for wait_seconds in RETRY_WAIT_SECONDS + [None]:
        try:
            return request_matrix(
                property_points, facility_points, api_key
            )
        except requests.HTTPError as error:
            status = error.response.status_code
            is_retryable = status in RETRY_STATUS_CODES
            if not is_retryable or wait_seconds is None:
                raise
            print(f"HTTP {status}, retrying in {wait_seconds} s")
            time.sleep(wait_seconds)

## Reduce one batch's matrix to each location's nearest candidate

In [22]:
def pick_nearest_candidate(candidate_distances, candidate_durations,
                            candidate_positions, facility_ids):
    """Return the nearest candidate's facility ID, distance and duration.

    Returns all-empty values if every candidate distance is NaN.
    """
    if np.isnan(candidate_distances).all():
        return None, np.nan, np.nan

    best = int(np.nanargmin(candidate_distances))
    best_position = candidate_positions[best]
    return (
        facility_ids[best_position],
        candidate_distances[best],
        candidate_durations[best],
    )

In [23]:
def summarise_batch(
    location_rows,
    facility_positions,
    distances,
    durations,
    candidates,
    facility_ids,
):
    """Reduce a batch's matrix to each location's nearest candidate.

    Each location is only compared with its own candidate facilities,
    not every facility in the batch.
    """
    column_of = {
        position: column
        for column, position in enumerate(facility_positions)
    }
    records = []

    for row, location in enumerate(location_rows):
        candidate_positions = candidates[location]
        columns = [
            column_of[position] for position in candidate_positions
        ]
        facility_id, distance_m, duration_s = pick_nearest_candidate(
            distances[row, columns],
            durations[row, columns],
            candidate_positions,
            facility_ids,
        )
        records.append({
            "location_row": location,
            "nearest_facility_id": facility_id,
            "road_distance_m": distance_m,
            "duration_s": duration_s,
        })

    return pd.DataFrame(records)

## Run a batch of requests, with caching

Each batch is saved to its own file as soon as it returns, so a run
that stops partway (daily quota, an error) can be resumed by rerunning
the same cell: already-saved batches are skipped.

In [24]:
def get_batch_points(location_rows, facility_positions, facility_coordinates):
    """Look up the [lon, lat] points for one batch's sources and
    destinations.
    """
    property_points = unique_locations.loc[
        location_rows, [RENTAL_LON_COLUMN, RENTAL_LAT_COLUMN]
    ].to_numpy()
    facility_points = facility_coordinates.loc[
        facility_positions, ["Longitude", "Latitude"]
    ].to_numpy()
    return property_points, facility_points

In [25]:
def send_one_batch(
    batch,
    api_key,
    candidates,
    facility_ids,
    facility_coordinates,
    cache_file,
):
    """Send one batch to ORS and save its summarised result.

    Returns "sent" on success, "quota" on HTTP 429, or "error" on any
    other failure (the error itself is printed).
    """
    location_rows, facility_positions = batch
    property_points, facility_points = get_batch_points(
        location_rows, facility_positions, facility_coordinates
    )

    try:
        distances, durations = request_matrix_with_retries(
            property_points, facility_points, api_key
        )
    except requests.HTTPError as error:
        status = error.response.status_code
        if status == 429:
            print("ORS quota reached (HTTP 429). Rerun later.")
            return "quota"
        print(f"HTTP {status}: {error.response.text[:200]}")
        return "error"

    batch_results = summarise_batch(
        location_rows,
        facility_positions,
        distances,
        durations,
        candidates,
        facility_ids,
    )
    batch_results.to_csv(cache_file, index=False)
    return "sent"

In [26]:
def run_pending_batches(
    batches,
    api_key,
    max_requests,
    candidates,
    cache_dir,
    facility_ids,
    facility_coordinates,
):
    """Send each uncached batch to ORS, up to max_requests this run.

    Returns True once every batch is cached, or False if the run
    stopped early (quota, an error, or the request limit for this run).
    """
    cache_dir.mkdir(exist_ok=True, parents=True)
    n_sent = 0

    for batch_number, batch in enumerate(batches):
        cache_file = cache_dir / f"batch_{batch_number:04d}.csv"
        if cache_file.exists():
            continue
        if n_sent >= max_requests:
            print(f"Paused after {n_sent} requests. Rerun to continue.")
            return False

        outcome = send_one_batch(
            batch, api_key, candidates, facility_ids,
            facility_coordinates, cache_file,
        )
        if outcome != "sent":
            return False

        n_sent += 1
        if n_sent % 25 == 0:
            print(f"{n_sent} requests sent")
        time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"All batches saved ({n_sent} requests sent this run).")
    return True

## First pass: nearest candidates by straight line, then route

Finds each location's `N_CANDIDATES` nearest facilities by straight-line
distance, routes to them, and marks a location **confirmed** when its
best road distance is at most the straight-line distance to its
farthest candidate (`d10_m`): nothing outside the candidates can then
be closer by road.

In [27]:
def find_nearest_candidates(tree, k):
    """Query the tree for each location's k nearest facilities.

    Returns the candidate facility positions and straight-line
    distances in metres.
    """
    distances_rad, positions = tree.query(location_radians, k=k)
    distances_m = distances_rad * EARTH_RADIUS_M
    return positions, distances_m

In [28]:
def load_cached_results(cache_dir):
    """Read and concatenate every batch result file in cache_dir."""
    cached_files = sorted(cache_dir.glob("batch_*.csv"))
    return pd.concat(
        (pd.read_csv(path) for path in cached_files), ignore_index=True
    )

In [29]:
def run_first_pass(facility_coordinates, tree, destination_name):
    """Run the first routing pass for one destination.

    Returns (location_results, processing_order), or (None, None) if
    the ORS run was interrupted and needs to be rerun later.
    """
    facility_ids = facility_coordinates["Facility ID"].to_numpy()
    k = min(N_CANDIDATES, len(facility_coordinates))
    candidate_positions, straight_line_m = find_nearest_candidates(
        tree, k
    )

    processing_order = sort_locations_into_strips(
        unique_locations, RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN,
        STRIP_DEGREES,
    )
    batches = build_batches(
        processing_order, candidate_positions, MAX_MATRIX_CELLS
    )
    print(f"First pass: {len(batches)} requests")

    cache_dir = OUTPUT_DIR / f"cache_{destination_name}_k{N_CANDIDATES}"
    completed = run_pending_batches(
        batches, ORS_API_KEY, MAX_REQUESTS_PER_RUN,
        candidate_positions, cache_dir, facility_ids,
        facility_coordinates,
    )
    if not completed:
        return None, None

    road_results = load_cached_results(cache_dir)
    location_results = unique_locations.join(
        road_results.set_index("location_row")
    )
    location_results["d10_m"] = straight_line_m[:, -1]
    location_results["is_confirmed"] = (
        location_results["road_distance_m"] <= location_results["d10_m"]
    )
    print(location_results["is_confirmed"].value_counts())
    return location_results, processing_order

## Second pass: wider search for unconfirmed locations

Retries only the unconfirmed locations with `N_CANDIDATES_SECOND_PASS`
candidates. Anything still unconfirmed afterwards is left as `NaN`,
rather than kept as an unproven distance, for manual investigation.

In [30]:
def get_unconfirmed_rows(location_results):
    """Return the location row numbers not yet confirmed."""
    is_unconfirmed = ~location_results["is_confirmed"].to_numpy(bool)
    return np.flatnonzero(is_unconfirmed)

In [31]:
def merge_second_pass_results(
    location_results, second_pass_results, wide_straight_line_m, k_second
):
    """Overwrite rerouted rows with their second-pass results.

    candidates_used and bound_m are only initialised if this is the
    first time this table has been through a wider pass; a later pass
    (e.g. a third) must not reset an already-widened bound back down.
    """
    merged = location_results.copy()
    if "candidates_used" not in merged.columns:
        merged["candidates_used"] = N_CANDIDATES
    if "bound_m" not in merged.columns:
        merged["bound_m"] = merged["d10_m"]

    rerouted_rows = second_pass_results.index.to_numpy()
    result_columns = [
        "nearest_facility_id", "road_distance_m", "duration_s"
    ]
    merged.loc[rerouted_rows, result_columns] = (
        second_pass_results[result_columns]
    )
    merged.loc[rerouted_rows, "candidates_used"] = k_second
    merged.loc[rerouted_rows, "bound_m"] = wide_straight_line_m[
        rerouted_rows, -1
    ]
    merged["is_confirmed"] = merged["road_distance_m"] <= merged["bound_m"]
    return merged

In [32]:
def null_out_unconfirmed(results, destination_name):
    """Blank the distance columns for rows still unconfirmed.

    Leaves these for manual investigation rather than keeping an
    unproven distance.
    """
    still_unconfirmed = ~results["is_confirmed"]
    n_nulled = still_unconfirmed.sum()
    columns_to_clear = [
        "nearest_facility_id", "road_distance_m", "duration_s"
    ]
    results.loc[still_unconfirmed, columns_to_clear] = np.nan
    print(f"{destination_name}: {n_nulled} locations nulled for review")
    return results

In [33]:
def run_second_pass(
    facility_coordinates,
    tree,
    destination_name,
    location_results,
    processing_order,
):
    """Run the second routing pass for one destination's unconfirmed
    locations.

    Returns the final per-location result table, or None if the ORS
    run was interrupted and needs to be rerun later.
    """
    unconfirmed_rows = get_unconfirmed_rows(location_results)
    if len(unconfirmed_rows) == 0:
        print(f"{destination_name}: all confirmed after first pass.")
        return location_results

    facility_ids = facility_coordinates["Facility ID"].to_numpy()
    k_second = min(N_CANDIDATES_SECOND_PASS, len(facility_coordinates))
    wide_positions, wide_straight_line_m = find_nearest_candidates(
        tree, k_second
    )

    is_unconfirmed = np.isin(processing_order, unconfirmed_rows)
    second_pass_order = processing_order[is_unconfirmed]
    second_pass_batches = build_batches(
        second_pass_order, wide_positions, MAX_MATRIX_CELLS
    )
    print(f"Second pass: {len(second_pass_batches)} requests")

    cache_dir = (
        OUTPUT_DIR
        / f"cache_{destination_name}_k{N_CANDIDATES_SECOND_PASS}"
    )
    completed = run_pending_batches(
        second_pass_batches, ORS_API_KEY, MAX_REQUESTS_PER_RUN,
        wide_positions, cache_dir, facility_ids, facility_coordinates,
    )
    if not completed:
        return None

    second_pass_results = load_cached_results(cache_dir).set_index(
        "location_row"
    )
    final_results = merge_second_pass_results(
        location_results, second_pass_results, wide_straight_line_m,
        k_second,
    )
    final_results = null_out_unconfirmed(final_results, destination_name)
    print(final_results["is_confirmed"].value_counts())
    return final_results

## Driver: run both passes for one destination type

In [34]:
def build_facility_tree(facility_coordinates):
    """Build a haversine BallTree over a destination's facilities."""
    facility_radians = np.radians(
        facility_coordinates[["Latitude", "Longitude"]].to_numpy()
    )
    return BallTree(facility_radians, metric="haversine")

In [35]:
def process_destination(destination_name, facilities_path):
    """Run the two-pass road-distance pipeline for one destination type.

    Returns the final per-location result table, or None if either
    pass was interrupted and needs to be rerun later.
    """
    print(f"\n=== {destination_name} ===")
    facility_coordinates = pd.read_csv(facilities_path)
    tree = build_facility_tree(facility_coordinates)

    location_results, processing_order = run_first_pass(
        facility_coordinates, tree, destination_name
    )
    if location_results is None:
        print(f"{destination_name}: first pass incomplete, rerun later.")
        return None

    results_path = OUTPUT_DIR / f"road_distances_{destination_name}.csv"
    location_results.to_csv(results_path, index=False)

    final_results = run_second_pass(
        facility_coordinates, tree, destination_name,
        location_results, processing_order,
    )
    if final_results is None:
        print(f"{destination_name}: second pass incomplete, rerun later.")
        return None

    final_results.to_csv(results_path, index=False)
    return final_results

## Run every destination type

Resumable: rerunning this cell continues any destination whose run was
interrupted, since already-cached batches are skipped.

In [39]:
all_destination_results = {}

for destination_name, facilities_path in DESTINATION_SETS.items():
    all_destination_results[destination_name] = process_destination(
        destination_name, facilities_path
    )


=== sports_rec ===
First pass: 162 requests
All batches saved (0 requests sent this run).
is_confirmed
True     10648
False      359
Name: count, dtype: int64
Second pass: 17 requests
All batches saved (0 requests sent this run).
sports_rec: 9 locations nulled for review
is_confirmed
True     10998
False        9
Name: count, dtype: int64

=== shopping_centre ===
First pass: 91 requests
All batches saved (0 requests sent this run).
is_confirmed
True     10829
False      178
Name: count, dtype: int64
Second pass: 6 requests
All batches saved (0 requests sent this run).
shopping_centre: 3 locations nulled for review
is_confirmed
True     11004
False        3
Name: count, dtype: int64

=== medical_doctors ===
First pass: 101 requests
All batches saved (0 requests sent this run).
is_confirmed
True     10634
False      373
Name: count, dtype: int64
Second pass: 8 requests
All batches saved (0 requests sent this run).
medical_doctors: 22 locations nulled for review
is_confirmed
True     109

In [36]:
for destination_name, facilities_path in DESTINATION_SETS.items():
    facility_coordinates = pd.read_csv(facilities_path)
    tree = build_facility_tree(facility_coordinates)
    k = min(N_CANDIDATES, len(facility_coordinates))
    candidate_positions, _ = find_nearest_candidates(tree, k)
    processing_order = sort_locations_into_strips(
        unique_locations, RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN, STRIP_DEGREES
    )
    expected_batches = len(
        build_batches(processing_order, candidate_positions, MAX_MATRIX_CELLS)
    )
    cache_dir = OUTPUT_DIR / f"cache_{destination_name}_k{N_CANDIDATES}"
    actual_batches = len(list(cache_dir.glob("batch_*.csv")))
    complete = expected_batches == actual_batches
    print(f"{destination_name}: {actual_batches}/{expected_batches} — {'OK' if complete else 'INCOMPLETE'}")

sports_rec: 162/162 — OK
shopping_centre: 91/91 — OK
medical_doctors: 101/101 — OK
medical_clinic: 109/109 — OK
hospital: 84/84 — OK
park_reserve: 99/99 — OK
park_national: 99/99 — OK
park_local: 248/248 — OK
school_primary: 129/129 — OK
school_secondary: 101/101 — OK


## Attach every destination's results to all properties

Each completed destination contributes four columns:
`nearest_<destination>_facility_id`, `<destination>_road_distance_m`,
`<destination>_duration_s`, `<destination>_is_confirmed`.

In [37]:
def build_location_lookup(unique_locations):
    """Map each rental property to its unique location's row number."""
    return unique_locations.rename_axis("location_row").reset_index()

In [38]:
def rename_destination_columns(results, destination_name):
    """Prefix a destination's result columns with its own name."""
    columns_to_keep = [
        "nearest_facility_id", "road_distance_m", "duration_s",
        "is_confirmed",
    ]
    return results[columns_to_keep].rename(columns={
        "nearest_facility_id": f"nearest_{destination_name}_facility_id",
        "road_distance_m": f"{destination_name}_road_distance_m",
        "duration_s": f"{destination_name}_duration_s",
        "is_confirmed": f"{destination_name}_is_confirmed",
    })

In [39]:
def attach_destination_results(rentals_with_locations, all_results):
    """Join every destination's renamed results onto the rentals table."""
    merged = rentals_with_locations
    for destination_name, results in all_results.items():
        if results is None:
            continue
        renamed = rename_destination_columns(results, destination_name)
        merged = merged.merge(
            renamed, left_on="location_row", right_index=True, how="left"
        )
    return merged

In [40]:
location_lookup = build_location_lookup(unique_locations)
rentals_with_locations = rentals_raw.merge(
    location_lookup,
    on=[RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN],
    how="left",
)
assert len(rentals_with_locations) == len(rentals_raw)

In [41]:
rentals_with_proximity = attach_destination_results(
    rentals_with_locations, all_destination_results
)
rentals_with_proximity = rentals_with_proximity.drop(columns="location_row")
assert len(rentals_with_proximity) == len(rentals_raw)

## Save the final table

In [42]:
OUTPUT_PATH = OUTPUT_DIR / f"rentals_with_all_proximity_{ORS_PROFILE}.parquet"
rentals_with_proximity.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved {len(rentals_with_proximity)} rows to {OUTPUT_PATH}")

Saved 12317 rows to proximity_results/rentals_with_all_proximity_driving-car.parquet


## Re-check nulls before resolving the remaining NaNs

Read the saved output back in and count nulls per destination's
`_road_distance_m` column.

In [43]:
FINAL_OUTPUT_PATH = OUTPUT_DIR / f"rentals_with_all_proximity_{ORS_PROFILE}.parquet"

rentals_check = pd.read_parquet(FINAL_OUTPUT_PATH)
print(f"{len(rentals_check)} rows")

distance_columns = [
    col for col in rentals_check.columns if col.endswith("_road_distance_m")
]
rentals_check[distance_columns].isna().sum()

12317 rows


sports_rec_road_distance_m           15
shopping_centre_road_distance_m       3
medical_doctors_road_distance_m      28
medical_clinic_road_distance_m       36
hospital_road_distance_m             39
park_reserve_road_distance_m         31
park_national_road_distance_m       794
park_local_road_distance_m           75
school_primary_road_distance_m        3
school_secondary_road_distance_m      1
dtype: int64

## Third pass: confirm the remaining nulled locations

For each destination, retry only the locations left as `NaN` after the
second pass, with `N_CANDIDATES_THIRD_PASS` candidates. Anything still
unconfirmed after this is left as `NaN` for manual investigation — this
is the last automatic widening; no further pass is run automatically.

In [44]:
def get_nulled_rows(location_results):
    """Return the location row numbers with no road distance recorded."""
    is_nulled = location_results["road_distance_m"].isna().to_numpy()
    return np.flatnonzero(is_nulled)

In [45]:
def run_third_pass(
    facility_coordinates,
    tree,
    destination_name,
    location_results,
    processing_order,
):
    """Run a third routing pass for locations still nulled after the
    second pass.

    Returns the final per-location result table, or None if the ORS
    run was interrupted and needs to be rerun later.
    """
    nulled_rows = get_nulled_rows(location_results)
    if len(nulled_rows) == 0:
        print(f"{destination_name}: nothing left to confirm.")
        return location_results

    facility_ids = facility_coordinates["Facility ID"].to_numpy()
    k_third = min(N_CANDIDATES_THIRD_PASS, len(facility_coordinates))
    wide_positions, wide_straight_line_m = find_nearest_candidates(
        tree, k_third
    )

    is_nulled = np.isin(processing_order, nulled_rows)
    third_pass_order = processing_order[is_nulled]
    third_pass_batches = build_batches(
        third_pass_order, wide_positions, MAX_MATRIX_CELLS
    )
    print(f"Third pass: {len(third_pass_batches)} requests")

    cache_dir = (
        OUTPUT_DIR / f"cache_{destination_name}_k{N_CANDIDATES_THIRD_PASS}"
    )
    completed = run_pending_batches(
        third_pass_batches, ORS_API_KEY, MAX_REQUESTS_PER_RUN,
        wide_positions, cache_dir, facility_ids, facility_coordinates,
    )
    if not completed:
        return None

    third_pass_results = load_cached_results(cache_dir).set_index(
        "location_row"
    )
    final_results = merge_second_pass_results(
        location_results, third_pass_results, wide_straight_line_m,
        k_third,
    )
    final_results = null_out_unconfirmed(final_results, destination_name)
    print(final_results["is_confirmed"].value_counts())
    return final_results

In [46]:
def rerun_third_pass_for_destination(destination_name, facilities_path):
    """Load a destination's saved results and run the third pass on it."""
    print(f"\n=== {destination_name} ===")
    facility_coordinates = pd.read_csv(facilities_path)
    tree = build_facility_tree(facility_coordinates)

    results_path = OUTPUT_DIR / f"road_distances_{destination_name}.csv"
    location_results = pd.read_csv(results_path)

    processing_order = sort_locations_into_strips(
        unique_locations, RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN,
        STRIP_DEGREES,
    )

    final_results = run_third_pass(
        facility_coordinates, tree, destination_name,
        location_results, processing_order,
    )
    if final_results is None:
        print(f"{destination_name}: third pass incomplete, rerun later.")
        return None

    final_results.to_csv(results_path, index=False)
    return final_results

### Run the third pass for every destination

Resumable, same as the earlier passes: rerunning this cell continues
any destination whose run was interrupted.

In [47]:
for destination_name, facilities_path in DESTINATION_SETS.items():
    updated_results = rerun_third_pass_for_destination(
        destination_name, facilities_path
    )
    if updated_results is not None:
        all_destination_results[destination_name] = updated_results


=== sports_rec ===
Third pass: 1 requests
All batches saved (0 requests sent this run).
sports_rec: 1 locations nulled for review
is_confirmed
True     11006
False        1
Name: count, dtype: int64

=== shopping_centre ===


Third pass: 1 requests
All batches saved (0 requests sent this run).
shopping_centre: 0 locations nulled for review
is_confirmed
True    11007
Name: count, dtype: int64

=== medical_doctors ===
Third pass: 2 requests
All batches saved (0 requests sent this run).
medical_doctors: 2 locations nulled for review
is_confirmed
True     11005
False        2
Name: count, dtype: int64

=== medical_clinic ===
Third pass: 2 requests
All batches saved (0 requests sent this run).
medical_clinic: 26 locations nulled for review
is_confirmed
True     10981
False       26
Name: count, dtype: int64

=== hospital ===
Third pass: 1 requests
All batches saved (0 requests sent this run).
hospital: 0 locations nulled for review
is_confirmed
True    11007
Name: count, dtype: int64

=== park_reserve ===
Third pass: 2 requests
All batches saved (0 requests sent this run).
park_reserve: 1 locations nulled for review
is_confirmed
True     11006
False        1
Name: count, dtype: int64

=== park_national ===
Third

## Final resolution: full search for all remaining nulls, with caching

Each location's full-search result is cached individually as soon as
it's resolved, so an error partway through only loses the one location
being processed when it happened — a rerun skips every location
already cached and continues from where it stopped.

In [48]:
def split_facilities_into_chunks(n_facilities, max_cells):
    """Split facility row numbers into chunks of at most max_cells."""
    n_chunks = int(np.ceil(n_facilities / max_cells))
    return np.array_split(np.arange(n_facilities), n_chunks)

In [49]:
def route_one_location_to_all_facilities(
    location_row, facility_coordinates, facility_chunks, api_key
):
    """Route one location against every facility, chunk by chunk.

    Returns the full arrays of distances and durations, in the same
    order as facility_coordinates.
    """
    property_point = unique_locations.loc[
        [location_row], [RENTAL_LON_COLUMN, RENTAL_LAT_COLUMN]
    ].to_numpy()

    distance_chunks = []
    duration_chunks = []
    for chunk in facility_chunks:
        facility_points = facility_coordinates.loc[
            chunk, ["Longitude", "Latitude"]
        ].to_numpy()
        distances, durations = request_matrix_with_retries(
            property_point, facility_points, api_key
        )
        distance_chunks.append(distances[0])
        duration_chunks.append(durations[0])
        time.sleep(SECONDS_BETWEEN_REQUESTS)

    return np.concatenate(distance_chunks), np.concatenate(duration_chunks)

In [50]:
def build_full_search_record(distances, durations, facility_ids):
    """Build a one-row result dict from a full-search routing result.

    found_route is False if every facility was unreachable, in which
    case the other fields are left empty.
    """
    if np.isnan(distances).all():
        return {
            "nearest_facility_id": None,
            "road_distance_m": np.nan,
            "duration_s": np.nan,
            "found_route": False,
        }

    best = int(np.nanargmin(distances))
    return {
        "nearest_facility_id": facility_ids[best],
        "road_distance_m": distances[best],
        "duration_s": durations[best],
        "found_route": True,
    }

In [51]:
def resolve_one_location(
    location_row, facility_coordinates, facility_chunks, facility_ids,
    cache_dir,
):
    """Resolve one location's full-search result, using the cache if
    it already exists. Always returns the cached or freshly-saved
    record as a one-row DataFrame.
    """
    cache_file = cache_dir / f"location_{location_row:06d}.csv"
    if cache_file.exists():
        return pd.read_csv(cache_file)

    distances, durations = route_one_location_to_all_facilities(
        location_row, facility_coordinates, facility_chunks, ORS_API_KEY
    )
    record = build_full_search_record(distances, durations, facility_ids)
    record["location_row"] = location_row

    record_df = pd.DataFrame([record])
    record_df.to_csv(cache_file, index=False)
    return record_df

In [52]:
def apply_full_search_record(location_results, record_row):
    """Write one cached full-search record into location_results."""
    location_row = int(record_row["location_row"])
    if not record_row["found_route"]:
        return False

    location_results.loc[location_row, "nearest_facility_id"] = (
        record_row["nearest_facility_id"]
    )
    location_results.loc[location_row, "road_distance_m"] = (
        record_row["road_distance_m"]
    )
    location_results.loc[location_row, "duration_s"] = (
        record_row["duration_s"]
    )
    location_results.loc[location_row, "candidates_used"] = (
        record_row["n_facilities"]
    )
    location_results.loc[location_row, "bound_m"] = np.nan
    location_results.loc[location_row, "is_confirmed"] = True
    return True

In [53]:
def resolve_nulls_with_full_search(destination_name, facilities_path):
    """Route every still-nulled location against every facility.

    Each location's result is cached individually, so a rerun after an
    error only redoes the locations not yet resolved.
    """
    facility_coordinates = pd.read_csv(facilities_path)
    facility_ids = facility_coordinates["Facility ID"].to_numpy()
    facility_chunks = split_facilities_into_chunks(
        len(facility_coordinates), MAX_MATRIX_CELLS
    )

    results_path = OUTPUT_DIR / f"road_distances_{destination_name}.csv"
    location_results = pd.read_csv(results_path)
    nulled_rows = get_nulled_rows(location_results)

    if len(nulled_rows) == 0:
        return location_results

    cache_dir = OUTPUT_DIR / f"cache_{destination_name}_full_search"
    cache_dir.mkdir(exist_ok=True, parents=True)
    print(f"{destination_name}: full search for {len(nulled_rows)} locations")

    for location_row in nulled_rows:
        record_df = resolve_one_location(
            location_row, facility_coordinates, facility_chunks,
            facility_ids, cache_dir,
        )
        record_row = record_df.iloc[0].copy()
        record_row["n_facilities"] = len(facility_coordinates)
        found_route = apply_full_search_record(location_results, record_row)
        if not found_route:
            print(f"  location {location_row}: no route found at all")

    location_results.to_csv(results_path, index=False)
    return location_results

### Run the full search for every destination

Resumable: rerunning this cell skips every location already cached and
only retries the ones that failed.

In [54]:
for destination_name, facilities_path in DESTINATION_SETS.items():
    updated_results = resolve_nulls_with_full_search(
        destination_name, facilities_path
    )
    all_destination_results[destination_name] = updated_results

sports_rec: full search for 1 locations
medical_doctors: full search for 2 locations
medical_clinic: full search for 26 locations
park_reserve: full search for 1 locations
park_national: full search for 22 locations
park_local: full search for 6 locations


## Attach the final results and confirm everything is resolved

Rebuild the rentals table from the fully-resolved per-destination
results, then check that no destination has any remaining nulls.

In [55]:
rentals_with_proximity = attach_destination_results(
    rentals_with_locations, all_destination_results
)
rentals_with_proximity = rentals_with_proximity.drop(columns="location_row")
assert len(rentals_with_proximity) == len(rentals_raw)

In [56]:
rentals_with_proximity.to_parquet(FINAL_OUTPUT_PATH, index=False)
print(f"Saved {len(rentals_with_proximity)} rows to {FINAL_OUTPUT_PATH}")

Saved 12317 rows to proximity_results/rentals_with_all_proximity_driving-car.parquet


### Confirm no nulls remain

Re-read the saved file from disk and check every destination's
distance column for nulls, independent of anything held in memory.

In [57]:
def check_no_remaining_nulls(rentals_path):
    """Print the null count for every destination's road distance
    column, read fresh from disk.
    """
    rentals_check = pd.read_parquet(rentals_path)
    distance_columns = [
        col for col in rentals_check.columns
        if col.endswith("_road_distance_m")
    ]
    null_counts = rentals_check[distance_columns].isna().sum()
    print(null_counts)
    return (null_counts == 0).all()


all_resolved = check_no_remaining_nulls(FINAL_OUTPUT_PATH)
assert all_resolved, "Some destinations still have unresolved nulls"
print("\nAll destinations fully resolved.")

sports_rec_road_distance_m          0
shopping_centre_road_distance_m     0
medical_doctors_road_distance_m     0
medical_clinic_road_distance_m      0
hospital_road_distance_m            0
park_reserve_road_distance_m        0
park_national_road_distance_m       0
park_local_road_distance_m          0
school_primary_road_distance_m      0
school_secondary_road_distance_m    0
dtype: int64

All destinations fully resolved.
